# Working

**Request:** [T04] Clean these orders up and total them

> New email from Sai Suram <sai@agents.agentstore.it.com>
> Subject: [T04] Clean these orders up and total them
> Thread ID: AAQkADI1N2Y5MTE3LTE1MDctNGY0Yy1iYzQ5LWEzNmE5NzAyYzk4NQAQACwKd7LRD7dInkXmx1nruNo=

Each cell below is one run in the sandbox, in the order it happened, with whatever it printed and produced.


In [1]:
import pandas as pd
import io

data = """order_id,customer,order_date,amount,qty
1001,Acme Corp,2026-01-15,"$1,250.00",5
1002,acme corp ,15/01/2026,"$980.50",3
1003,Beta Ltd,2026-02-03,"$2,100.00",8
1003,Beta Ltd,2026-02-03,"$2,100.00",8
1004,BETA LTD,03/02/2026,"$450.25",2
1005,Gamma Inc,2026-02-20,"$1,875.00",-4
1006,Gamma Inc,20/02/2026,"$730.00",1"""

df = pd.read_csv(io.StringIO(data))

# Standardize customer names
df['customer'] = df['customer'].str.title().replace({'Acme Corp ': 'Acme Corp', 'Beta Ltd ': 'Beta Ltd', 'Gamma Inc ': 'Gamma Inc'})

# Convert amount to numeric
df['amount'] = df['amount'].replace({'\$': ''}, regex=True).astype(float)

# Convert order_date to datetime and standardize format
df['order_date'] = pd.to_datetime(df['order_date'], dayfirst=True)

# Remove duplicate entries based on order_id and other columns (assuming same order_id, customer, order_date, amount, qty is a full duplicate)
df_cleaned = df.drop_duplicates(subset=['order_id', 'customer', 'order_date', 'amount', 'qty'])

# Handle negative quantity: Assuming a negative quantity indicates an error and should be treated as positive for revenue calculation
df_cleaned['qty'] = df_cleaned['qty'].abs()

# Calculate total price for each order line
df_cleaned['total_price'] = df_cleaned['amount'] * df_cleaned['qty']

# Calculate revenue by customer
revenue_by_customer = df_cleaned.groupby('customer')['total_price'].sum().reset_index()
revenue_by_customer.rename(columns={'total_price': 'total_revenue'}, inplace=True)

# Prepare for Excel output
output_filename = '/tmp/output/cleaned_orders_and_revenue.xlsx'
writer = pd.ExcelWriter(output_filename, engine='xlsxwriter')
df_cleaned.to_excel(writer, sheet_name='Cleaned Orders', index=False)
revenue_by_customer.to_excel(writer, sheet_name='Revenue by Customer', index=False)
writer.close()

print(f"Cleaned data:
{df_cleaned.to_markdown(index=False)}\n")
print(f"Revenue by customer:
{revenue_by_customer.to_markdown(index=False)}\n")
print(f"Output file: {output_filename}")

<string>:24: SyntaxWarning: invalid escape sequence '\$'
  File "<string>", line 49
    print(f"Cleaned data:
          ^
SyntaxError: unterminated f-string literal (detected at line 49)



[this step exited with code 1 — it did not finish]


In [2]:
import pandas as pd
import re
import io

# Data provided in the email
data = """order_id,customer,order_date,amount,qty
1001,Acme Corp,2026-01-15,"1,250.00",5
1002,acme corp ,15/01/2026,"980.50",3
1003,Beta Ltd,2026-02-03,"2,100.00",8
1003,Beta Ltd,2026-02-03,"2,100.00",8
1004,BETA LTD,03/02/2026,"450.25",2
1005,Gamma Inc,2026-02-20,"1,875.00",-4
1006,Gamma Inc,20/02/2026,"730.00",1"""

df = pd.read_csv(io.StringIO(data))

fixes = []

# 1. Clean customer names (normalize case, remove extra spaces)
original_customers = df['customer'].tolist()
df['customer'] = df['customer'].str.strip().str.title()
if original_customers != df['customer'].tolist():
    fixes.append("Normalized customer names (e.g., 'acme corp' to 'Acme Corp').")

# 2. Clean 'amount' column (remove '$' and commas, convert to numeric)
def clean_amount(amount_str):
    if isinstance(amount_str, str):
        return float(amount_str.replace('$', '').replace(',', ''))
    return float(amount_str)

original_amounts = df['amount'].tolist()
df['amount'] = df['amount'].apply(clean_amount)
if original_amounts != df['amount'].tolist():
    fixes.append("Cleaned 'amount' column by removing '$' and commas, then converted to numeric.")

# 3. Handle 'order_date' column (convert to consistent datetime format)
def parse_date(date_str):
    for fmt in ('%Y-%m-%d', '%d/%m/%Y'):
        try:
            return pd.to_datetime(date_str, format=fmt)
        except ValueError:
            continue
    return pd.NaT # Not a Time

original_dates = df['order_date'].tolist()
df['order_date'] = df['order_date'].apply(parse_date)
if df['order_date'].isnull().any():
    fixes.append("Handled inconsistent date formats in 'order_date' column.")

# 4. Remove duplicate rows (based on all columns for exact duplicates)
initial_rows = len(df)
df.drop_duplicates(inplace=True)
if len(df) < initial_rows:
    fixes.append(f"Removed {initial_rows - len(df)} duplicate rows.")

# 5. Handle negative quantities (assume they are errors and set to 0 or absolute value, or flag them)
# For this analysis, we'll assume a negative quantity means it was entered incorrectly
# and that the actual quantity should be the absolute value, but we will note it.
negative_qty_rows = df[df['qty'] < 0]
if not negative_qty_rows.empty:
    fixes.append(f"Corrected {len(negative_qty_rows)} rows with negative quantities to their absolute value. Original negative quantities were: {negative_qty_rows['qty'].tolist()}.")
    df['qty'] = df['qty'].abs()

# Calculate total revenue per order
df['total_revenue'] = df['amount'] * df['qty']

# Calculate revenue by customer
revenue_by_customer = df.groupby('customer')['total_revenue'].sum().reset_index()
revenue_by_customer.rename(columns={'total_revenue': 'Total Revenue'}, inplace=True)

# Prepare output for Excel
output_df = df[['order_id', 'customer', 'order_date', 'amount', 'qty', 'total_revenue']]
output_df['order_date'] = output_df['order_date'].dt.strftime('%Y-%m-%d') # Format date for Excel export

# Save to Excel
output_file_path = '/tmp/output/cleaned_orders_and_revenue.xlsx'
with pd.ExcelWriter(output_file_path, engine='openpyxl') as writer:
    output_df.to_excel(writer, sheet_name='Cleaned Orders', index=False)
    revenue_by_customer.to_excel(writer, sheet_name='Revenue by Customer', index=False)

print("--- Fixes Made ---")
for fix in fixes:
    print(f"- {fix}")
print("\n--- Revenue by Customer ---")
print(revenue_by_customer.to_string(index=False))


--- Fixes Made ---
- Normalized customer names (e.g., 'acme corp' to 'Acme Corp').
- Cleaned 'amount' column by removing '$' and commas, then converted to numeric.
- Removed 1 duplicate rows.
- Corrected 1 rows with negative quantities to their absolute value. Original negative quantities were: [-4].

--- Revenue by Customer ---
 customer  Total Revenue
Acme Corp         9191.5
 Beta Ltd        17700.5
Gamma Inc         8230.0



[files written: cleaned_orders_and_revenue.xlsx]
